In [2]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/cases_train2 (12).xlsx')
df['Оценка'] = df['Оценка'].round().astype('Int64')
df_test_cases = pd.read_excel('/content/Test1_2 (3).xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].round().astype('Int64')
#df = pd.read_excel("/content/good table.xlsx")
df.head(100)


,Кейс,Решение,Решение кейса,ЦА,Проработка решения,Финансовая модель и метрики,Анализ рисков,Доказательства,Оценка
0,1,1,Я выбрал отрасль туризма и гостиничного бизнес...,1,1,1,1,1,1
1,1,2,Я выбрал для разработки отраслевого решения сф...,2,2,2,2,2,2
2,1,4,"Я выбрал отрасль образования, а именно сегмент...",2,4,4,3,3,3
3,1,5,Я выбрал отрасль розничной торговли продуктами...,1,3,2,2,1,2
4,1,6,Я выбрал для отраслевого решения сферу гостини...,3,2,2,2,3,2
...,...,...,...,...,...,...,...,...,...
95,1,97,Я выбрал отрасль грузоперевозок. Целевая аудит...,2,2,4,3,1,2
96,1,98,Я выбрал отрасль ремонта техники. Целевая ауди...,3,2,3,2,2,2
97,1,99,Я выбрал отрасль такси. Целевая аудитория: вод...,1,1,4,3,1,2
98,1,100,Я выбрал отрасль ремонта квартир. Целевая ауди...,3,3,3,2,2,3


In [3]:
df["Оценка"].value_counts()

,count
Оценка,
3,606
4,453
2,381
1,250
5,211


In [5]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor
import pickle

In [7]:
y_audience = df['ЦА'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_audience, X_test_text_audience, y_train_audience, y_test_audience = train_test_split(
    X_text, y_audience, test_size=0.2, random_state=42
)
tfidf_audience = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_audience = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_audience = tfidf_audience.fit_transform(X_train_text_audience)
X_train_svd_audience = svd_audience.fit_transform(X_train_tfidf_audience)
X_test_tfidf_audience = tfidf_audience.transform(X_test_text_audience)
X_test_svd_audience = svd_audience.transform(X_test_tfidf_audience)

In [8]:
from sklearn.model_selection import GridSearchCV

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [10]:
from catboost import CatBoostClassifier

In [11]:
model_audience = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)

In [12]:
model_audience.fit(X_train_svd_audience , y_train_audience)

0:	learn: 1.5687226	total: 113ms	remaining: 1m 53s
50:	learn: 1.1632517	total: 1.56s	remaining: 29.1s
100:	learn: 0.9756004	total: 3.96s	remaining: 35.3s
150:	learn: 0.8493415	total: 5.93s	remaining: 33.4s
200:	learn: 0.7584601	total: 7.35s	remaining: 29.2s
250:	learn: 0.6848634	total: 8.78s	remaining: 26.2s
300:	learn: 0.6217717	total: 10.2s	remaining: 23.7s
350:	learn: 0.5711149	total: 11.7s	remaining: 21.6s
400:	learn: 0.5258994	total: 13.1s	remaining: 19.6s
450:	learn: 0.4870307	total: 14.5s	remaining: 17.7s
500:	learn: 0.4568455	total: 16.5s	remaining: 16.4s
550:	learn: 0.4245726	total: 18.7s	remaining: 15.3s
600:	learn: 0.3991902	total: 20.2s	remaining: 13.4s
650:	learn: 0.3750953	total: 21.6s	remaining: 11.6s
700:	learn: 0.3541959	total: 23.1s	remaining: 9.84s
750:	learn: 0.3345829	total: 24.5s	remaining: 8.13s
800:	learn: 0.3170281	total: 26s	remaining: 6.45s
850:	learn: 0.3014628	total: 27.4s	remaining: 4.79s
900:	learn: 0.2862220	total: 29.2s	remaining: 3.2s
950:	learn: 0.272

CatBoostClassifier(auto_class_weights='Balanced', depth=3, early_stopping_rounds=50, eval_metric='MultiClass', iterations=1000, learning_rate=0.1, loss_function='MultiClass', random_seed=42, verbose=50)

In [13]:
y_pred_audience = model_audience.predict(X_test_svd_audience)
print(classification_report(y_test_audience, y_pred_audience))
print("MAE=",mean_absolute_error(y_test_audience, y_pred_audience))

              precision    recall  f1-score   support

           1       0.78      0.80      0.79        56
           2       0.67      0.64      0.66        90
           3       0.58      0.45      0.51        84
           4       0.59      0.63      0.61        76
           5       0.71      0.84      0.77        75

    accuracy                           0.66       381
   macro avg       0.66      0.67      0.67       381
weighted avg       0.66      0.66      0.66       381

MAE= 0.4304461942257218


In [14]:
y_pred_audience_train = model_audience.predict(X_train_svd_audience)
print(classification_report(y_train_audience, y_pred_audience_train))
print("MAE=",mean_absolute_error(y_train_audience, y_pred_audience_train))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00       188
           2       1.00      1.00      1.00       331
           3       1.00      1.00      1.00       339
           4       1.00      1.00      1.00       364
           5       1.00      1.00      1.00       298

    accuracy                           1.00      1520
   macro avg       1.00      1.00      1.00      1520
weighted avg       1.00      1.00      1.00      1520

MAE= 0.0013157894736842105


In [15]:
y_sol= df['Проработка решения'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_sol, X_test_text_sol, y_train_sol, y_test_sol = train_test_split(
    X_text, y_sol, test_size=0.2, random_state=42
)
tfidf_sol = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_sol= TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_sol = tfidf_sol.fit_transform(X_train_text_sol)
X_train_svd_sol = svd_sol.fit_transform(X_train_tfidf_sol)
X_test_tfidf_sol = tfidf_sol.transform(X_test_text_sol)
X_test_svd_sol = svd_sol.transform(X_test_tfidf_sol)

In [16]:
model_sol = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_sol.fit(X_train_svd_sol , y_train_sol)
y_pred_sol = model_sol.predict(X_test_svd_sol)
#y_pred_sol
print(classification_report(y_test_sol, y_pred_sol))
print("MAE=",mean_absolute_error(y_test_sol, y_pred_sol))

0:	learn: 1.5730430	total: 125ms	remaining: 2m 4s
50:	learn: 1.1808643	total: 3.17s	remaining: 59s
100:	learn: 0.9978820	total: 7.61s	remaining: 1m 7s
150:	learn: 0.8720386	total: 10.6s	remaining: 59.7s
200:	learn: 0.7754617	total: 12.1s	remaining: 48.1s
250:	learn: 0.7000307	total: 13.5s	remaining: 40.4s
300:	learn: 0.6346122	total: 15s	remaining: 34.8s
350:	learn: 0.5797806	total: 16.4s	remaining: 30.4s
400:	learn: 0.5350912	total: 18.1s	remaining: 27.1s
450:	learn: 0.4959335	total: 20.6s	remaining: 25.1s
500:	learn: 0.4618240	total: 22.1s	remaining: 22s
550:	learn: 0.4314338	total: 23.6s	remaining: 19.2s
600:	learn: 0.4051296	total: 25s	remaining: 16.6s
650:	learn: 0.3817942	total: 26.4s	remaining: 14.2s
700:	learn: 0.3566263	total: 27.9s	remaining: 11.9s
750:	learn: 0.3368416	total: 29.3s	remaining: 9.72s
800:	learn: 0.3190555	total: 30.7s	remaining: 7.63s
850:	learn: 0.3015634	total: 33.3s	remaining: 5.83s
900:	learn: 0.2863863	total: 35s	remaining: 3.84s
950:	learn: 0.2712327	tot

In [17]:
y_pred_sol_train = model_sol.predict(X_train_svd_sol)
#y_pred_sol
print(classification_report(y_train_sol, y_pred_sol_train))
print("MAE=",mean_absolute_error(y_train_sol, y_pred_sol_train))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00       185
           2       1.00      0.99      1.00       363
           3       0.99      0.99      0.99       351
           4       0.99      1.00      0.99       338
           5       1.00      1.00      1.00       283

    accuracy                           1.00      1520
   macro avg       1.00      1.00      1.00      1520
weighted avg       1.00      1.00      1.00      1520

MAE= 0.0059210526315789476


In [18]:
y_finance= df['Финансовая модель и метрики'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_finance, X_test_text_finance,y_train_finance, y_test_finance = train_test_split(
    X_text, y_finance, test_size=0.2, random_state=42
)
tfidf_finance = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_finance = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_finance = tfidf_finance.fit_transform(X_train_text_finance)
X_train_svd_finance = svd_finance.fit_transform(X_train_tfidf_finance)
X_test_tfidf_finance = tfidf_finance.transform(X_test_text_finance)
X_test_svd_finance = svd_finance.transform(X_test_tfidf_finance)

In [19]:
model_finance = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_finance.fit(X_train_svd_finance , y_train_finance)
y_pred_finance = model_finance.predict(X_test_svd_finance)
print(classification_report(y_test_finance, y_pred_finance))
print("MAE=",mean_absolute_error(y_test_finance, y_pred_finance))

0:	learn: 1.5637397	total: 59.5ms	remaining: 17.8s
50:	learn: 1.1181441	total: 2.1s	remaining: 10.3s
100:	learn: 0.9413358	total: 3.59s	remaining: 7.07s
150:	learn: 0.8201607	total: 5.07s	remaining: 5.01s
200:	learn: 0.7214544	total: 6.6s	remaining: 3.25s
250:	learn: 0.6506749	total: 8.07s	remaining: 1.58s
299:	learn: 0.5953560	total: 9.51s	remaining: 0us
              precision    recall  f1-score   support

           1       0.68      0.83      0.75        59
           2       0.57      0.56      0.56        97
           3       0.64      0.51      0.57        92
           4       0.63      0.71      0.67        75
           5       0.75      0.72      0.74        58

    accuracy                           0.64       381
   macro avg       0.65      0.67      0.66       381
weighted avg       0.64      0.64      0.64       381

MAE= 0.4435695538057743


In [20]:
y_pred_finance_train = model_finance.predict(X_train_svd_finance)
print(classification_report(y_train_finance, y_pred_finance_train))
print("MAE=",mean_absolute_error(y_train_finance, y_pred_finance_train))

              precision    recall  f1-score   support

           1       0.90      0.98      0.94       204
           2       0.92      0.87      0.89       389
           3       0.89      0.85      0.87       384
           4       0.89      0.87      0.88       344
           5       0.85      0.97      0.91       199

    accuracy                           0.89      1520
   macro avg       0.89      0.91      0.90      1520
weighted avg       0.89      0.89      0.89      1520

MAE= 0.1394736842105263


In [24]:
y_risks= df['Анализ рисков'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_risks, X_test_text_risks, y_train_risks, y_test_risks = train_test_split(
    X_text, y_risks, test_size=0.2, random_state=42
)
tfidf_risks = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_risks = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_risks = tfidf_risks.fit_transform(X_train_text_risks)
X_train_svd_risks = svd_risks.fit_transform(X_train_tfidf_risks)
X_test_tfidf_risks = tfidf_risks.transform(X_test_text_risks)
X_test_svd_risks = svd_risks.transform(X_test_tfidf_risks)

In [25]:
model_risks = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_risks.fit(X_train_svd_risks , y_train_risks)
y_pred_risks = model_risks.predict(X_test_svd_risks)
print(classification_report(y_test_risks, y_pred_risks))
print("MAE=",mean_absolute_error(y_test_risks, y_pred_risks))

0:	learn: 1.5603281	total: 94.8ms	remaining: 1m 34s
50:	learn: 1.1459153	total: 1.92s	remaining: 35.7s
100:	learn: 0.9441748	total: 3.39s	remaining: 30.1s
150:	learn: 0.8172605	total: 4.89s	remaining: 27.5s
200:	learn: 0.7255668	total: 6.39s	remaining: 25.4s
250:	learn: 0.6529575	total: 7.86s	remaining: 23.4s
300:	learn: 0.5937429	total: 9.3s	remaining: 21.6s
350:	learn: 0.5462625	total: 10.8s	remaining: 19.9s
400:	learn: 0.5072954	total: 13.3s	remaining: 19.8s
450:	learn: 0.4721867	total: 15.1s	remaining: 18.4s
500:	learn: 0.4401309	total: 16.5s	remaining: 16.5s
550:	learn: 0.4143740	total: 17.9s	remaining: 14.6s
600:	learn: 0.3889408	total: 19.4s	remaining: 12.9s
650:	learn: 0.3647166	total: 20.9s	remaining: 11.2s
700:	learn: 0.3448642	total: 22.3s	remaining: 9.51s
750:	learn: 0.3263899	total: 23.7s	remaining: 7.87s
800:	learn: 0.3075141	total: 26.2s	remaining: 6.5s
850:	learn: 0.2908471	total: 28s	remaining: 4.91s
900:	learn: 0.2765595	total: 29.5s	remaining: 3.24s
950:	learn: 0.262

In [26]:
y_pred_risks_train = model_risks.predict(X_train_svd_risks)
print(classification_report(y_train_risks, y_pred_risks_train))
print("MAE=",mean_absolute_error(y_train_risks, y_pred_risks_train))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00       240
           2       0.99      0.99      0.99       380
           3       1.00      0.99      1.00       374
           4       1.00      1.00      1.00       330
           5       0.99      1.00      1.00       196

    accuracy                           1.00      1520
   macro avg       1.00      1.00      1.00      1520
weighted avg       1.00      1.00      1.00      1520

MAE= 0.0059210526315789476


In [27]:
y_proves= df['Доказательства'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_proves, X_test_text_proves, y_train_proves, y_test_proves = train_test_split(
    X_text, y_proves, test_size=0.2, random_state=42
)
tfidf_proves = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_proves = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_proves = tfidf_proves.fit_transform(X_train_text_proves)
X_train_svd_proves = svd_proves.fit_transform(X_train_tfidf_proves)
X_test_tfidf_proves = tfidf_proves.transform(X_test_text_proves)
X_test_svd_proves = svd_proves.transform(X_test_tfidf_proves)

In [28]:
model_proves = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_proves.fit(X_train_svd_proves , y_train_proves)
y_pred_proves = model_proves.predict(X_test_svd_proves)
print(classification_report(y_test_proves, y_pred_proves))
print("MAE=",mean_absolute_error(y_test_proves, y_pred_proves))

0:	learn: 1.5693879	total: 126ms	remaining: 2m 5s
50:	learn: 1.1733227	total: 4.11s	remaining: 1m 16s
100:	learn: 0.9888114	total: 7.28s	remaining: 1m 4s
150:	learn: 0.8569643	total: 9.77s	remaining: 55s
200:	learn: 0.7663980	total: 11.3s	remaining: 45s
250:	learn: 0.6902366	total: 12.8s	remaining: 38.2s
300:	learn: 0.6268514	total: 14.3s	remaining: 33.1s
350:	learn: 0.5729304	total: 16.9s	remaining: 31.2s
400:	learn: 0.5289651	total: 18.5s	remaining: 27.7s
450:	learn: 0.4905918	total: 20s	remaining: 24.3s
500:	learn: 0.4575694	total: 21.5s	remaining: 21.4s
550:	learn: 0.4262272	total: 23s	remaining: 18.7s
600:	learn: 0.3985749	total: 24.5s	remaining: 16.2s
650:	learn: 0.3743865	total: 25.9s	remaining: 13.9s
700:	learn: 0.3516416	total: 27.6s	remaining: 11.8s
750:	learn: 0.3321980	total: 30.2s	remaining: 10s
800:	learn: 0.3141397	total: 31.8s	remaining: 7.91s
850:	learn: 0.2968235	total: 33.3s	remaining: 5.83s
900:	learn: 0.2819231	total: 34.8s	remaining: 3.83s
950:	learn: 0.2687410	to

In [29]:
y_pred_proves_train = model_proves.predict(X_train_svd_proves)
print(classification_report(y_train_proves, y_pred_proves_train))
print("MAE=",mean_absolute_error(y_train_proves, y_pred_proves_train))

              precision    recall  f1-score   support

           1       1.00      1.00      1.00       238
           2       1.00      1.00      1.00       387
           3       1.00      1.00      1.00       306
           4       0.99      0.99      0.99       346
           5       0.99      1.00      0.99       243

    accuracy                           1.00      1520
   macro avg       1.00      1.00      1.00      1520
weighted avg       1.00      1.00      1.00      1520

MAE= 0.003289473684210526


In [ ]:
new_results_df = pd.DataFrame(columns=['Id','Текст_решения','Анализ ЦА','Проработка решения','Финансовая модель и метрики','Анализ рынков','Доказательства','Предсказанная_оценка','Дата'])

In [ ]:
import re

In [ ]:
dec = '''МОЙ ПРОЕКТ

Я хочу сделать бота для школьников. Он будет помогать решать задачи.

Целевая аудитория - школьники. Им это нужно для учебы.

Мое решение - бот в телеграме. Он будет бесплатный. Похожих ботов нет.

Финансы: разработка стоит примерно 500 тысяч рублей. Потом будем зарабатывать на рекламе.

Риски: могут появиться конкуренты. Будем делать лучше.

Доказательства: я сам учился в школе и знаю, что это нужно. Многие мои друзья тоже так думают.'''

In [30]:
def get_prediction_audience(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_audience.transform([text])
  text_svd = svd_audience.transform(text_tfidf)
  score = model_audience.predict(text_svd)[0]
  return int(score)

In [31]:
def get_prediction_solution(text):
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_sol.transform([text])
  text_svd = svd_sol.transform(text_tfidf)
  score = model_sol.predict(text_svd)[0]
  return int(score)

In [32]:
def get_prediction_finance(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_finance.transform([text])
  text_svd = svd_finance.transform(text_tfidf)
  score = model_finance.predict(text_svd)[0]
  return int(score)

In [33]:
def get_prediction_risks(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_risks.transform([text])
  text_svd = svd_risks.transform(text_tfidf)
  score = model_risks.predict(text_svd)[0]
  return int(score)

In [34]:
def get_prediction_proves(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_proves.transform([text])
  text_svd = svd_proves.transform(text_tfidf)
  score = model_proves.predict(text_svd)[0]
  return int(score)

In [35]:
df_test_cases['pred_audience'] = df_test_cases['Решение кейса'].apply(get_prediction_audience)
df_test_cases['pred_sol'] = df_test_cases['Решение кейса'].apply(get_prediction_solution)
df_test_cases['pred_finance'] = df_test_cases['Решение кейса'].apply(get_prediction_finance)
df_test_cases['pred_risks'] = df_test_cases['Решение кейса'].apply(get_prediction_risks)
df_test_cases['pred_proves'] = df_test_cases['Решение кейса'].apply(get_prediction_proves)

/tmp/ipykernel_2959/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_2959/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_2959/131350308.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_2959/2249487333.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a

In [36]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)


In [37]:
#ЦА
accuracy_audience = accuracy_score(df_test_cases['ЦА'], df_test_cases['pred_audience'])
f1micro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='micro')
f1macro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='macro')
print('accuracy_audience= ', accuracy_audience)
print('f1micro_audience= ', f1micro_audience)
print('f1macro_audience= ', f1macro_audience)
print(classification_report(df_test_cases['ЦА'], df_test_cases['pred_audience']))
print("MAE=", mean_absolute_error(df_test_cases['ЦА'], df_test_cases['pred_audience']))

accuracy_audience=  0.5615384615384615
f1micro_audience=  0.5615384615384615
f1macro_audience=  0.5466666666666666
              precision    recall  f1-score   support

           1       0.56      0.61      0.58        23
           2       0.50      0.27      0.35        26
           3       0.55      0.62      0.58        34
           4       0.61      0.50      0.55        22
           5       0.57      0.80      0.67        25

    accuracy                           0.56       130
   macro avg       0.56      0.56      0.55       130
weighted avg       0.56      0.56      0.55       130

MAE= 0.6


In [38]:
accuracy_sol = accuracy_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'])
f1micro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='micro')
f1macro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='macro')
print('accuracy_sol= ', accuracy_sol)
print('f1micro_sol= ', f1micro_sol)
print('f1macro_sol= ', f1macro_sol)
print(classification_report(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))
print("MAE=", mean_absolute_error(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))

accuracy_sol=  0.5230769230769231
f1micro_sol=  0.5230769230769231
f1macro_sol=  0.5081541116943301
              precision    recall  f1-score   support

           1       0.56      0.95      0.70        21
           2       0.60      0.19      0.29        32
           3       0.61      0.37      0.46        30
           4       0.49      0.68      0.57        25
           5       0.45      0.64      0.53        22

    accuracy                           0.52       130
   macro avg       0.54      0.56      0.51       130
weighted avg       0.55      0.52      0.49       130

MAE= 0.6307692307692307


In [39]:
accuracy_finance = accuracy_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'])
f1micro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='micro')
f1macro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='macro')
print('accuracy_finance= ', accuracy_finance)
print('f1micro_finance= ', f1micro_finance)
print('f1macro_finance= ', f1macro_finance)
print(classification_report(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))
print("MAE=", mean_absolute_error(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))

accuracy_finance=  0.5076923076923077
f1micro_finance=  0.5076923076923077
f1macro_finance=  0.48660350212245473
              precision    recall  f1-score   support

           1       0.55      0.91      0.69        23
           2       0.65      0.39      0.49        33
           3       0.47      0.52      0.49        31
           4       0.35      0.50      0.41        24
           5       1.00      0.21      0.35        19

    accuracy                           0.51       130
   macro avg       0.61      0.51      0.49       130
weighted avg       0.59      0.51      0.49       130

MAE= 0.5923076923076923


In [40]:
accuracy_risks = accuracy_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'])
f1micro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='micro')
f1macro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='macro')
print('accuracy_risks= ', accuracy_risks)
print('f1micro_risks= ', f1micro_risks)
print('f1macro_risks= ', f1macro_risks)
print(classification_report(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))
print("MAE=", mean_absolute_error(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))

accuracy_risks=  0.5076923076923077
f1micro_risks=  0.5076923076923077
f1macro_risks=  0.46273572411420716
              precision    recall  f1-score   support

           1       0.58      0.69      0.63        26
           2       0.50      0.28      0.36        32
           3       0.50      0.57      0.53        30
           4       0.44      0.74      0.56        27
           5       1.00      0.13      0.24        15

    accuracy                           0.51       130
   macro avg       0.61      0.48      0.46       130
weighted avg       0.56      0.51      0.48       130

MAE= 0.6


In [41]:
accuracy_proves = accuracy_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'])
f1micro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='micro')
f1macro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='macro')
print('accuracy_proves= ', accuracy_proves)
print('f1micro_proves= ', f1micro_proves)
print('f1macro_proves= ', f1macro_proves)
print(classification_report(df_test_cases['Доказательства'], df_test_cases['pred_proves']))
print("MAE", mean_absolute_error(df_test_cases['Доказательства'], df_test_cases['pred_proves']))


accuracy_proves=  0.45384615384615384
f1micro_proves=  0.45384615384615384
f1macro_proves=  0.43986650863951093
              precision    recall  f1-score   support

           1       0.45      0.78      0.57        18
           2       0.63      0.50      0.56        38
           3       0.38      0.24      0.29        25
           4       0.30      0.42      0.35        19
           5       0.46      0.40      0.43        30

    accuracy                           0.45       130
   macro avg       0.44      0.47      0.44       130
weighted avg       0.47      0.45      0.45       130

MAE 0.8


In [42]:
sec_dec = """АНАЛИЗ ЦА:
Выделено 3 сегмента:
- Школьники 10-11 класс (45%)
- Студенты (35%)
- Учителя (20%)
Проведен опрос 100 человек.

РЕШЕНИЕ:
Экосистема из 2 продуктов: телеграм бот + дашборд для учителей.
Партнеры: Яндекс.Образование, МФТИ.

ФИНАНСЫ:
CAPEX: 2 млн руб. CAC = 100 руб, LTV = 2000 руб.
Окупаемость: 10 месяцев.

РИСКИ:
Технические (30%) - резервный сервер.

ДОКАЗАТЕЛЬСТВА:
Пилот в школе: NPS = 65.
"""

In [43]:
df_test_cases["Predict_score"] = round((df_test_cases['pred_audience'] + df_test_cases['pred_sol'] + df_test_cases['pred_finance'] + df_test_cases['pred_risks'] + df_test_cases['pred_proves'])/5)

In [44]:
print("MAE= ",mean_absolute_error(df_test_cases["Оценка"], df_test_cases["Predict_score"]))

MAE=  0.5


In [ ]:
sol_test = """ Я выбрала отрасль кофеен и кофе-точек с собой (кофе на вынос, кофейные киоски, небольшие кофейни на 2–5 столиков). ЦА разделена на две группы. Первая группа — это микробизнес: кофе-байки и кофе-островки с одним сотрудником, работают как ИП или самозанятые. Их проблема в том, что кофейные зерна нужно покупать свежей обжарки каждую неделю, а хорошие обжарщики требуют предоплату 100% за партию от 5 кг, это около 20–30 тысяч рублей единовременно, что для маленькой точки с ежедневной выручкой 5–7 тысяч рублей чувствительно. Вторая группа — это малый бизнес: кофейни с посадочными местами и штатом 2–5 бариста. Их проблема в том, что сезонность спроса сильно различается: зимой продажи выше за счет горячих напитков, а летом люди покупают холодный кофе и чаще берут с собой, но в межсезонье (апрель и октябрь) падение выручки достигает 30%, при этом аренду и зарплату платить надо. Я опиралась на данные исследования «Рынок кофеен России 2024» от компании CoffeeData: рост рынка на 15% за год, количество кофеен достигло 12 тысяч, из них 70% — малый и микробизнес. Также я посмотрел открытую статистику по кофейному рынку на сайте Росстата и несколько статей в профильных телеграм-каналах. В качестве решения я предлагаю продукт «Альфа.Кофе»: кредит на закупку зёрен с отсрочкой первого платежа на 30 дней и с льготным периодом 0% на первые две недели, а также сезонный овердрафт на покрытие аренды в межсезонье на сумму до 100 тысяч рублей. Партнеры — два крупных обжарщика зерна «Coffe Lab» и «Torrefacto», с которыми можно договориться о более выгодных ценах для клиентов банка. Отличие от конкурентов в том, что ни у Сбера, ни у Т-Банка нет специального продукта для кофеен с отсрочкой именно под зерно. По финансам: разработка кредитного продукта обойдется примерно в 6 миллионов рублей, интеграция с партнерами-обжарщиками — в 2 миллиона, маркетинг в кофейных чатах и через конференции бариста — в 2 миллиона. Прогноз: 350 клиентов в первый год, средний доход с клиента — 20 тысяч рублей (за счет процентов по кредиту и эквайринга), выручка — 7 миллионов рублей. Окупаемость — примерно 2,5 года. Риски: конкурентный — другие банки могут запустить похожие продукты; кредитный — часть клиентов может не вернуть деньги, но мы будем проверять по выписке с кофемашины, кто сколько продает. В доказательство я использовал данные CoffeeData и результаты пары интервью с владельцами кофеен в своем городе."""

In [ ]:
predicted_score_audience = get_prediction_audience(sec_dec)
predicted_score_sol = get_prediction_solution(sec_dec)
predicted_score_finance = get_prediction_finance(sec_dec)
predicted_score_risks = get_prediction_risks(sec_dec)
predicted_score_proves = get_prediction_proves(sec_dec)
predicted_score_audience2 = get_prediction_audience(sec_dec)
predicted_score_sol2 = get_prediction_solution(sec_dec)
predicted_score_finance2 = get_prediction_finance(sec_dec)
predicted_score_risks2 = get_prediction_risks(sec_dec)
predicted_score_proves2 = get_prediction_proves(sec_dec)
new_row = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience,
    'Проработка решения': predicted_score_sol,
    'Финансовая модель и метрики': predicted_score_finance,
    'Анализ рынков': predicted_score_risks,
    'Доказательства': predicted_score_proves,
    'Предсказанная_оценка': round((predicted_score_audience + predicted_score_sol +
                      predicted_score_finance + predicted_score_risks +
                      predicted_score_proves) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_row2 = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience2,
    'Проработка решения': predicted_score_sol2,
    'Финансовая модель и метрики': predicted_score_finance2,
    'Анализ рынков': predicted_score_risks2,
    'Доказательства': predicted_score_proves2,
    'Предсказанная_оценка': round((predicted_score_audience2 + predicted_score_sol2 +
                      predicted_score_finance2 + predicted_score_risks2 +
                      predicted_score_proves2) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_results_df = pd.concat([new_results_df, new_row, new_row2], ignore_index=True)
new_results_df.to_excel("новая_таблица.xlsx", index=False)


/tmp/ipykernel_3876/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_3876/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)


In [ ]:
new_results_df

,Id,Текст_решения,Анализ ЦА,Проработка решения,Финансовая модель и метрики,Анализ рынков,Доказательства,Предсказанная_оценка,Дата
0,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
1,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
